In [1]:
import os
import sys

# Third-party numerical and data handling
import numpy as np
import pandas as pd
# Visualization
import matplotlib.pyplot as plt

# PyTorch core
import torch
from torch import nn
from torch.utils.data import DataLoader

from model_evaluation_helpers import check_device, read_preprocessed_images, ECGDataset, val_transforms, MultiHeadEfficientNet, get_probs_and_labels, compute_ranking_metrics
from sklearn.model_selection import train_test_split

from torchvision import models
from model_evaluation_helpers import Head

In [2]:
# Check and get device
device = check_device()

try: 
    print(image_set["train_000000.png"])
except Exception as e:
    image_set = read_preprocessed_images("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/proc_images.h5")
    

label_df = pd.read_csv("/Users/griffinfarrow/Documents/Data_Science_Projects/bhf_classification/data/train_final.csv", index_col=0)

image_names = list(image_set.keys())
image_ids = list([int(image_name.split(".")[0][-6:]) for image_name in image_names])
label_df = label_df.loc[label_df.index.isin(set(image_ids))]

# need to order label_df so that it has the same ordering as image_names
image_df = pd.DataFrame({
    "image_name": image_set.keys(),
})
image_df["image_id"] = image_df["image_name"].str.split(".", expand=True)[0].str[-6:].astype(int)
image_df = image_df.sort_values(by="image_id")
image_df = image_df.set_index("image_id")
# ensure ordering of label_df and image_df
label_df = label_df.loc[image_df.index]
X_train, X_test, y_train, y_test = train_test_split(image_df,
                                                    label_df,
                                                    test_size=0.2,
                                                    random_state=42,
                                                    shuffle=True,
                                                    stratify=label_df[["CD", "MI", "AF", "STTC", "HYP"]]
                                                    )

train_image_names = X_train["image_name"].tolist()
test_image_names = X_test["image_name"].tolist()

val_dataset = ECGDataset(
    image_names = test_image_names,
    image_set = image_set,
    labels_df = y_test,
    transforms = val_transforms
)

num_workers = 0 if sys.platform == 'darwin' else 4 
print(f"Using num_workers = {num_workers}")

val_dataloader = DataLoader( 
                        val_dataset,
                        batch_size=32, 
                        shuffle=False, 
                        num_workers=num_workers, 
                        pin_memory=True if device.type == "cuda" else False)

✓ MPS (Apple Silicon GPU) available

Selected device: mps
Using num_workers = 0


In [3]:
def evaluate_model(modelpath, modeltype, verbose=True):
    checkpoint = torch.load(modelpath, map_location=torch.device("cpu"), weights_only=False)
    current_epoch = checkpoint["epoch"]
    if verbose:
        print(f"Loading from checkpoint, last run epoch was {current_epoch}")
        
    model = MultiHeadEfficientNet(
        num_conditions=5, 
        hidden_dim=512, 
        dropout_rate=0.3, 
        model=modeltype
    ).to(device)

    model.load_state_dict(checkpoint["model_state_dict"])
    val_probs, val_labels = get_probs_and_labels(loader=val_dataloader, model=model, device=device)
    ranking_metrics = compute_ranking_metrics(val_labels, val_probs)
    
    return ranking_metrics

In [4]:
def evaluate_multiple_models(modeldict, modeltype="convnext"):
    """
    Takes in dictionary with format {str: str}, representing {modelname: path to model checkpoint} and collects ranking metrics
    for those models
    """
    collected_results = {}
    for modelname, modelpath in modeldict.items(): 
        results = evaluate_model(modelpath, modeltype)
        collected_results[modelname] = results
    return collected_results

In [8]:
basename = os.path.join(os.path.expanduser("~"), "Library", "CloudStorage", "OneDrive-Nexus365", "BHF_Cardiac_Problem", "Results_Collection")

In [9]:
path = os.path.join(basename, "Model_BCE_Unweighted", "best_model.pth")
convnext_backbone_metrics = evaluate_model(modelpath=path, modeltype="convnext")

Loading from checkpoint, last run epoch was 12


In [10]:
path = os.path.join(basename, "Model_BCE_EfficientNet_Backbone", "best_model.pth")
efficientnet_backbone_metrics = evaluate_model(modelpath=path, modeltype="efficientnet")

Loading from checkpoint, last run epoch was 13


In [11]:
# No shared feature process layer

class MultiHeadEfficientNet_1(nn.Module):
    def __init__(self, num_conditions=5, hidden_dim=512, dropout_rate=0.3, model="convnext"):
        super().__init__()
        
        if model=="convnext":
            backbone = models.convnext_base(weights="IMAGENET1K_V1", progress=True)
            in_features = backbone.classifier[2].in_features
        elif model=="efficientnet":
            backbone = models.efficientnet_v2_s(weights="DEFAULT")
            in_features = backbone.classifier[1].in_features
        else: 
            raise Exception(f"Model {model} not recognised ") 
        assert isinstance(in_features, int), f"in_features should be int, got {type(in_features)}"
        
        backbone.classifier[2] = nn.Linear(in_features, 5)
        self.backbone = backbone
        
        self.shared_feature_processor = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.BatchNorm1d(in_features),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_dim, num_conditions)
        )

        self.heads = nn.ModuleList([
            Head(hidden_dim, hidden_dim // 2, dropout_rate)
            for _ in range(num_conditions)
        ])
        
    def forward(self, x):
        backbone_feats = self.backbone(x).flatten(1) # squeeze non-batch dimension
        #processed_feats = self.shared_feature_processor(backbone_feats)
        #outputs = [head(processed_feats) for head in self.heads]
        return backbone_feats#torch.cat(outputs, dim=1)  # [batch, num_conditions]
    
modelpath =  os.path.join(basename, "Model_No_Feature_Process_Layer", "best_model.pth")
checkpoint = torch.load(modelpath, map_location=torch.device("cpu"), weights_only=False)
current_epoch = checkpoint["epoch"]

model = MultiHeadEfficientNet_1(
    num_conditions=5, 
    hidden_dim=512, 
    dropout_rate=0.3, 
    model="convnext"
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
val_probs, val_labels = get_probs_and_labels(loader=val_dataloader, model=model, device=device)
no_shared_feature_proc_ranking_metrics = compute_ranking_metrics(val_labels, val_probs)


In [12]:
# No separate classification heads

class MultiHeadEfficientNet_2(nn.Module):
    def __init__(self, num_conditions=5, hidden_dim=512, dropout_rate=0.3, model="convnext"):
        super().__init__()
        
        if model=="convnext":
            backbone = models.convnext_base(weights="IMAGENET1K_V1", progress=True)
            in_features = backbone.classifier[2].in_features
        elif model=="efficientnet":
            backbone = models.efficientnet_v2_s(weights="DEFAULT")
            in_features = backbone.classifier[1].in_features
        else: 
            raise Exception(f"Model {model} not recognised ") 
        assert isinstance(in_features, int), f"in_features should be int, got {type(in_features)}"
        backbone.classifier = nn.Identity()
        self.backbone = backbone
        
        self.shared_feature_processor = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.BatchNorm1d(in_features),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_dim, num_conditions)
        )

        self.heads = nn.ModuleList([
            Head(hidden_dim, hidden_dim // 2, dropout_rate)
            for _ in range(num_conditions)
        ])
        
    def forward(self, x):
        backbone_feats = self.backbone(x).flatten(1) # squeeze non-batch dimension
        processed_feats = self.shared_feature_processor(backbone_feats)
        #outputs = [head(processed_feats) for head in self.heads]
        return processed_feats#torch.cat(outputs, dim=1)  # [batch, num_conditions]
    
modelpath =  os.path.join(basename, "Model_No_Separate_Heads", "best_model.pth")
checkpoint = torch.load(modelpath, map_location=torch.device("cpu"), weights_only=False)
current_epoch = checkpoint["epoch"]

model = MultiHeadEfficientNet_2(
    num_conditions=5, 
    hidden_dim=512, 
    dropout_rate=0.3, 
    model="convnext"
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
val_probs, val_labels = get_probs_and_labels(loader=val_dataloader, model=model, device=device)
no_separate_heads_proc_ranking_metrics = compute_ranking_metrics(val_labels, val_probs)

In [13]:
# Large shared feature layer

class MultiHeadEfficientNet_3(nn.Module):
    def __init__(self, num_conditions=5, hidden_dim=1024, dropout_rate=0.3):
        super().__init__()

        backbone = models.convnext_base(weights="IMAGENET1K_V1", progress=True)
        in_features = backbone.classifier[2].in_features
        assert isinstance(in_features, int), f"in_features should be int, got {type(in_features)}"

        backbone.classifier = nn.Identity()
        self.backbone = backbone

        self.shared_feature_processor = nn.Sequential(
            nn.Linear(in_features, in_features),
            nn.BatchNorm1d(in_features),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(in_features, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.BatchNorm1d(hidden_dim//2),
            nn.GELU(),
            nn.Dropout(p=dropout_rate)
        )

        self.heads = nn.ModuleList([
            Head(hidden_dim // 2, hidden_dim // 4, dropout_rate)
            for _ in range(num_conditions)
        ])

    def forward(self, x):
        backbone_feats = self.backbone(x).flatten(1) # squeeze non-batch dimension
        processed_feats = self.shared_feature_processor(backbone_feats)
        outputs = [head(processed_feats) for head in self.heads]
        return torch.cat(outputs, dim=1)  # [batch, num_conditions]
    
modelpath =  os.path.join(basename, "Model_Large_Feature_Layer", "best_model.pth")
checkpoint = torch.load(modelpath, map_location=torch.device("cpu"), weights_only=False)
current_epoch = checkpoint["epoch"]

model = MultiHeadEfficientNet_3(
    num_conditions=5, 
    hidden_dim=512, 
    dropout_rate=0.3, 
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
val_probs, val_labels = get_probs_and_labels(loader=val_dataloader, model=model, device=device)
larger_shared_layer_proc_ranking_metrics = compute_ranking_metrics(val_labels, val_probs)

In [14]:
agg_results = {
            "ConvNext_Backbone": convnext_backbone_metrics, 
            "Efficientnet_Backbone": efficientnet_backbone_metrics, 
            "No_Shared_Feature_Proc": no_shared_feature_proc_ranking_metrics, 
            "No_Separate_Heads": no_separate_heads_proc_ranking_metrics, 
            "Larger_Shared_Feature_Proc": larger_shared_layer_proc_ranking_metrics}

In [15]:
from tabulate import tabulate
models = agg_results.keys()
to_write = []
for model in models: 
    to_write.append([model, 
                    agg_results[model]["macro_f1"],
                    agg_results[model]["macro_precision"], 
                    agg_results[model]["macro_recall"], 
                    agg_results[model]["macro_ece"], 
                    agg_results[model]["micro_ece"],
                    agg_results[model]["macro_brier"], 
                    agg_results[model]["macro_auroc"], 
                    agg_results[model]["micro_auroc"], 
                    agg_results[model]["macro_ap"], 
                    agg_results[model]["micro_ap"]])
    
print(tabulate(to_write, headers=["Model Name", "Macro F1", "Macro_Precision", "Macro_Recall", "Macro_ECE", "Micro_ECE", "Macro_Brier", "Macro_AUROC", "Micro_AUROC", "Macro_AP", "Micro_AP"]))

Model Name                    Macro F1    Macro_Precision    Macro_Recall    Macro_ECE    Micro_ECE    Macro_Brier    Macro_AUROC    Micro_AUROC    Macro_AP    Micro_AP
--------------------------  ----------  -----------------  --------------  -----------  -----------  -------------  -------------  -------------  ----------  ----------
ConvNext_Backbone             0.759605           0.755307        0.766016    0.0270267    0.0130162      0.0664135       0.937002       0.941537    0.820583    0.822527
Efficientnet_Backbone         0.735298           0.750681        0.726748    0.0198875    0.0143114      0.0687134       0.92953        0.935575    0.804013    0.810016
No_Shared_Feature_Proc        0.748177           0.746169        0.753403    0.0258196    0.0139496      0.0668794       0.93423        0.939036    0.810021    0.818047
No_Separate_Heads             0.7529             0.755344        0.753666    0.0238565    0.0119261      0.0659644       0.935556       0.940714    0.82141

In [16]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "f1"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name                    STTC_f1    HYP_f1     MI_f1     CD_f1     AF_f1
--------------------------  ---------  --------  --------  --------  --------
ConvNext_Backbone            0.757358  0.660574  0.753111  0.773266  0.853717
Efficientnet_Backbone        0.741026  0.623881  0.743448  0.745964  0.822171
No_Shared_Feature_Proc       0.756114  0.627566  0.750797  0.778502  0.827907
No_Separate_Heads            0.755862  0.64497   0.751416  0.757794  0.85446
Larger_Shared_Feature_Proc   0.754889  0.639148  0.759848  0.775244  0.834862


In [17]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "precision"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name                    STTC_precision    HYP_precision    MI_precision    CD_precision    AF_precision
--------------------------  ----------------  ---------------  --------------  --------------  --------------
ConvNext_Backbone                   0.721945         0.638889        0.751634        0.820467        0.843602
Efficientnet_Backbone               0.693878         0.696667        0.78343         0.79529         0.784141
No_Shared_Feature_Proc              0.727735         0.685897        0.729864        0.792703        0.794643
No_Separate_Heads                   0.757953         0.712418        0.721886        0.757188        0.827273
Larger_Shared_Feature_Proc          0.717472         0.629921        0.736453        0.789386        0.791304


In [18]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "recall"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name                    STTC_recall    HYP_recall    MI_recall    CD_recall    AF_recall
--------------------------  -------------  ------------  -----------  -----------  -----------
ConvNext_Backbone                0.796424      0.683784     0.754593       0.7312     0.864078
Efficientnet_Backbone            0.795048      0.564865     0.707349       0.7024     0.864078
No_Shared_Feature_Proc           0.786795      0.578378     0.772966       0.7648     0.864078
No_Separate_Heads                0.753783      0.589189     0.783465       0.7584     0.883495
Larger_Shared_Feature_Proc       0.796424      0.648649     0.784777       0.7616     0.883495


In [19]:
# compare per-label scores
models = agg_results.keys()
labels = ["STTC", "HYP", "MI", "CD", "AF"]
metric = "auroc"
labels = [f"{label}_{metric}" for label in labels]
label = f"per_label_{metric}"
to_write = []

for model in models: 
    to_write.append([model, *agg_results[model][label]])
print(tabulate(to_write, headers=["Model_Name", *labels]))

Model_Name                    STTC_auroc    HYP_auroc    MI_auroc    CD_auroc    AF_auroc
--------------------------  ------------  -----------  ----------  ----------  ----------
ConvNext_Backbone               0.937228     0.912378    0.919328    0.927805    0.988272
Efficientnet_Backbone           0.925543     0.899435    0.915441    0.921794    0.985439
No_Shared_Feature_Proc          0.936322     0.909801    0.918525    0.927459    0.979044
No_Separate_Heads               0.930437     0.90696     0.923379    0.931976    0.985029
Larger_Shared_Feature_Proc      0.936152     0.903275    0.925416    0.935449    0.988869
